# Day 66 — MLflow + Weights & Biases (W&B)
### Experiment tracking, model registry, artifact logging, hyperparameter sweeps

## 1. Setup

In [1]:
import sys
!{sys.executable} -m pip install mlflow wandb scikit-learn pandas numpy --quiet
print("Setup complete")

Setup complete



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Why Experiment Tracking Matters

In [2]:
problem_statement = """
WITHOUT EXPERIMENT TRACKING (what most beginners do):
======================================================
  model_v1.pkl          -- what hyperparameters? what accuracy?
  model_v2.pkl          -- was this better than v1?
  model_final.pkl       -- final compared to what?
  model_final_FINAL.pkl -- ...
  model_best.pkl        -- best on which metric?

  6 months later: "Which model is in production and why?"
  Answer: nobody knows.

WITH EXPERIMENT TRACKING:
=========================
  Run 001: lr=0.01, max_depth=3, accuracy=0.821, AUC=0.876
  Run 002: lr=0.01, max_depth=5, accuracy=0.834, AUC=0.891  <-- best
  Run 003: lr=0.001, max_depth=5, accuracy=0.819, AUC=0.863

  6 months later: "Which model is in production and why?"
  Answer: Run 002, AUC=0.891, promoted to production on 2026-01-15.
"""

print(problem_statement)

import pandas as pd
tools = pd.DataFrame([
    {"tool": "MLflow", "type": "Open-source", "hosting": "Self-hosted or managed",
     "best_for": "Model registry, deployment integration, self-hosted control"},
    {"tool": "Weights & Biases (W&B)", "type": "SaaS (free tier available)",
     "hosting": "Cloud (wandb.ai)", "best_for": "Rich visualisations, hyperparameter sweeps, team collaboration"},
    {"tool": "Neptune.ai", "type": "SaaS", "hosting": "Cloud",
     "best_for": "Large-scale experiment management"},
    {"tool": "Comet ML", "type": "SaaS", "hosting": "Cloud or self-hosted",
     "best_for": "NLP and computer vision teams"},
])
tools


WITHOUT EXPERIMENT TRACKING (what most beginners do):
  model_v1.pkl          -- what hyperparameters? what accuracy?
  model_v2.pkl          -- was this better than v1?
  model_final.pkl       -- final compared to what?
  model_final_FINAL.pkl -- ...
  model_best.pkl        -- best on which metric?

  6 months later: "Which model is in production and why?"
  Answer: nobody knows.

WITH EXPERIMENT TRACKING:
  Run 001: lr=0.01, max_depth=3, accuracy=0.821, AUC=0.876
  Run 002: lr=0.01, max_depth=5, accuracy=0.834, AUC=0.891  <-- best
  Run 003: lr=0.001, max_depth=5, accuracy=0.819, AUC=0.863

  6 months later: "Which model is in production and why?"
  Answer: Run 002, AUC=0.891, promoted to production on 2026-01-15.



,tool,type,hosting,best_for
0,MLflow,Open-source,Self-hosted or managed,"Model registry, deployment integration, self-h..."
1,Weights & Biases (W&B),SaaS (free tier available),Cloud (wandb.ai),"Rich visualisations, hyperparameter sweeps, te..."
2,Neptune.ai,SaaS,Cloud,Large-scale experiment management
3,Comet ML,SaaS,Cloud or self-hosted,NLP and computer vision teams


## 3. MLflow — Experiment Tracking

In [3]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.datasets import make_classification

# create a synthetic Titanic-like dataset
np.random.seed(42)
X, y = make_classification(
    n_samples=891, n_features=8, n_informative=5,
    n_redundant=2, random_state=42
)
feature_names = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_enc", "Embarked_enc", "Title_enc"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# set up MLflow experiment
mlflow.set_experiment("titanic-survival-prediction")

def train_and_log(model, model_name, params):
    with mlflow.start_run(run_name=model_name):
        # log hyperparameters
        mlflow.log_params(params)

        # train
        model.fit(X_train, y_train)

        # evaluate
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": round(accuracy_score(y_test, y_pred), 4),
            "auc": round(roc_auc_score(y_test, y_prob), 4),
            "f1": round(f1_score(y_test, y_pred), 4),
        }

        # log metrics
        mlflow.log_metrics(metrics)

        # log model
        mlflow.sklearn.log_model(model, "model")

        print(f"{model_name}: accuracy={metrics['accuracy']} | AUC={metrics['auc']} | F1={metrics['f1']}")
        return metrics

# run 3 experiments
print("Running MLflow experiments...\n")

train_and_log(
    LogisticRegression(C=1.0, max_iter=1000),
    "LogisticRegression-C1",
    {"model_type": "logistic_regression", "C": 1.0, "max_iter": 1000}
)

train_and_log(
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "RandomForest-100trees-depth5",
    {"model_type": "random_forest", "n_estimators": 100, "max_depth": 5}
)

train_and_log(
    RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    "RandomForest-200trees-depth8",
    {"model_type": "random_forest", "n_estimators": 200, "max_depth": 8}
)

print("\nAll runs logged to MLflow!")
print("View UI with: mlflow ui (then open http://localhost:5000)")

2026/07/13 16:55:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/13 16:55:35 INFO mlflow.store.db.utils: Updating database tables
2026/07/13 16:55:38 INFO mlflow.tracking.fluent: Experiment with name 'titanic-survival-prediction' does not exist. Creating a new experiment.


Running MLflow experiments...



2026/07/13 16:55:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression-C1: accuracy=0.7709 | AUC=0.8656 | F1=0.7784


2026/07/13 16:56:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RandomForest-100trees-depth5: accuracy=0.8827 | AUC=0.945 | F1=0.884


2026/07/13 16:56:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RandomForest-200trees-depth8: accuracy=0.8994 | AUC=0.9534 | F1=0.9032

All runs logged to MLflow!
View UI with: mlflow ui (then open http://localhost:5000)


## 4. MLflow Model Registry Concepts

In [4]:
registry_concepts = """
MLflow MODEL REGISTRY
=====================

After experiments are run, the best model can be promoted to the registry:

STAGES:
  None        -> just logged, not registered
  Staging     -> candidate for production, being tested
  Production  -> currently serving live traffic
  Archived    -> replaced, kept for audit trail

WORKFLOW:
  1. Run experiments (what we just did)
  2. Compare runs in MLflow UI
  3. Register best model:
       mlflow.register_model("runs:/<run_id>/model", "TitanicSurvivalPredictor")
  4. Transition to Staging:
       client.transition_model_version_stage(
           name="TitanicSurvivalPredictor", version=1, stage="Staging"
       )
  5. Test in staging environment
  6. Promote to Production:
       client.transition_model_version_stage(
           name="TitanicSurvivalPredictor", version=1, stage="Production"
       )

LOADING A PRODUCTION MODEL:
  model = mlflow.sklearn.load_model("models:/TitanicSurvivalPredictor/Production")

This is how the FastAPI service from Day 65 would load its model in production --
not from a local .pkl file but from the registry, ensuring full traceability.
"""

print(registry_concepts)

# show all runs from our experiment
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("titanic-survival-prediction")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc DESC"]
)

print("\nExperiment Runs (sorted by AUC):")
print(f"{'Run Name':<35} {'Accuracy':>10} {'AUC':>8} {'F1':>8}")
print("-" * 65)
for run in runs:
    name = run.data.tags.get("mlflow.runName", "unnamed")
    acc = run.data.metrics.get("accuracy", 0)
    auc = run.data.metrics.get("auc", 0)
    f1 = run.data.metrics.get("f1", 0)
    print(f"{name:<35} {acc:>10} {auc:>8} {f1:>8}")


MLflow MODEL REGISTRY

After experiments are run, the best model can be promoted to the registry:

STAGES:
  None        -> just logged, not registered
  Staging     -> candidate for production, being tested
  Production  -> currently serving live traffic
  Archived    -> replaced, kept for audit trail

WORKFLOW:
  1. Run experiments (what we just did)
  2. Compare runs in MLflow UI
  3. Register best model:
       mlflow.register_model("runs:/<run_id>/model", "TitanicSurvivalPredictor")
  4. Transition to Staging:
       client.transition_model_version_stage(
           name="TitanicSurvivalPredictor", version=1, stage="Staging"
       )
  5. Test in staging environment
  6. Promote to Production:
       client.transition_model_version_stage(
           name="TitanicSurvivalPredictor", version=1, stage="Production"
       )

LOADING A PRODUCTION MODEL:
  model = mlflow.sklearn.load_model("models:/TitanicSurvivalPredictor/Production")

This is how the FastAPI service from Day 65 would

## 5. Weights & Biases (W&B) — Richer Experiment Tracking

In [5]:
import wandb

wandb_intro = """
W&B SETUP (one-time):
=====================
1. Create free account at wandb.ai
2. Run: wandb login
3. Paste your API key from wandb.ai/settings

W&B vs MLflow:
  MLflow:  Open-source, self-hosted, better model registry + deployment integration
  W&B:     SaaS, richer visualisations, better hyperparameter sweeps, better team UX

Today we'll demonstrate W&B's sweep feature -- automated hyperparameter search.
"""
print(wandb_intro)

# W&B sweep configuration
sweep_config = {
    "name": "titanic-rf-sweep",
    "method": "grid",  # try all combinations
    "metric": {"name": "auc", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"values": [50, 100, 200]},
        "max_depth": {"values": [3, 5, 8]},
        "min_samples_split": {"values": [2, 5]},
    }
}

print("W&B Sweep Configuration:")
print(f"  Method: {sweep_config['method']}")
print(f"  Optimising: {sweep_config['metric']['name']} ({sweep_config['metric']['goal']})")
print(f"  Parameters:")
for param, config in sweep_config["parameters"].items():
    print(f"    {param}: {config['values']}")

total_runs = 1
for config in sweep_config["parameters"].values():
    total_runs *= len(config["values"])
print(f"\n  Total combinations: {total_runs} runs")
print(f"  (3 n_estimators x 3 max_depth x 2 min_samples_split)")


W&B SETUP (one-time):
1. Create free account at wandb.ai
2. Run: wandb login
3. Paste your API key from wandb.ai/settings

W&B vs MLflow:
  MLflow:  Open-source, self-hosted, better model registry + deployment integration
  W&B:     SaaS, richer visualisations, better hyperparameter sweeps, better team UX

Today we'll demonstrate W&B's sweep feature -- automated hyperparameter search.

W&B Sweep Configuration:
  Method: grid
  Optimising: auc (maximize)
  Parameters:
    n_estimators: [50, 100, 200]
    max_depth: [3, 5, 8]
    min_samples_split: [2, 5]

  Total combinations: 18 runs
  (3 n_estimators x 3 max_depth x 2 min_samples_split)


## 6. Hyperparameter Sweep — Simulated Locally

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import itertools

# simulate the sweep locally (no W&B account needed)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 8],
    "min_samples_split": [2, 5],
}

# generate all combinations
keys = list(param_grid.keys())
values = list(param_grid.values())
combinations = list(itertools.product(*values))

print(f"Running {len(combinations)} hyperparameter combinations...\n")
print(f"{'Run':<5} {'n_est':>6} {'depth':>6} {'min_split':>10} {'Accuracy':>10} {'AUC':>8} {'F1':>8}")
print("-" * 60)

results = []
for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    result = {
        **params,
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "auc": round(roc_auc_score(y_test, y_prob), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
    }
    results.append(result)
    print(f"{i+1:<5} {params['n_estimators']:>6} {params['max_depth']:>6} "
          f"{params['min_samples_split']:>10} {result['accuracy']:>10} "
          f"{result['auc']:>8} {result['f1']:>8}")

# find best
results_df = pd.DataFrame(results)
best = results_df.loc[results_df["auc"].idxmax()]
print(f"\nBest configuration (by AUC):")
print(f"  n_estimators={int(best['n_estimators'])}, max_depth={int(best['max_depth'])}, "
      f"min_samples_split={int(best['min_samples_split'])}")
print(f"  AUC={best['auc']} | Accuracy={best['accuracy']} | F1={best['f1']}")

Running 18 hyperparameter combinations...

Run    n_est  depth  min_split   Accuracy      AUC       F1
------------------------------------------------------------
1         50      3          2      0.838    0.923    0.838
2         50      3          5      0.838    0.923    0.838
3         50      5          2     0.8939   0.9434   0.8962
4         50      5          5     0.8771   0.9407   0.8778
5         50      8          2     0.8939   0.9486    0.895
6         50      8          5     0.8994    0.949   0.9011
7        100      3          2     0.8492   0.9241   0.8457
8        100      3          5     0.8492   0.9242   0.8457
9        100      5          2     0.8827    0.945    0.884
10       100      5          5     0.8771   0.9424   0.8778
11       100      8          2     0.8939   0.9547   0.8973
12       100      8          5     0.8939   0.9514   0.8962
13       200      3          2     0.8436   0.9272   0.8427
14       200      3          5     0.8436   0.9277   0.8

## 7. Summary — What We Covered Today

| Feature | MLflow | W&B |
|---------|--------|-----|
| Type | Open-source, self-hosted | SaaS, managed cloud |
| Experiment tracking | ✅ mlflow.log_params/metrics | ✅ wandb.log() |
| Model registry | ✅ Built-in (None/Staging/Production/Archived) | ⚠️ Limited in free tier |
| Hyperparameter sweeps | ⚠️ Manual or via Optuna | ✅ Built-in sweep agent |
| Visualisations | Basic charts | Rich interactive dashboards |
| Team collaboration | Self-managed | Built-in sharing |
| Cost | Free (self-hosted) | Free tier, paid for teams |
| Best for | Production deployment integration | Research, sweeps, team sharing |

**Key insights from today:**
- MLflow run comparison: RandomForest-200trees-depth8 won (AUC=0.9534 vs 0.8656 for LR)
- Hyperparameter sweep: 100 trees at depth 8 slightly beat 200 trees (AUC=0.9547 vs 0.9534)
- More trees doesn't always win — depth matters more than count for this dataset
- The model registry connects experiment tracking to deployment (Day 65 FastAPI service)